In [5]:
# Cell 1: Imports and Setup
from transformers import pipeline
import torch
import re

# Define model path
MODEL_PATH = "/Users/farooqkhan/My Mac/GitHub/LLMs/polymath-1/fine_tuned_model"

# Device setup for MPS (Apple Silicon) or CPU
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print("Using device:", device)

Using device: mps


In [29]:
# Cell 2: 
def evaluate_scientific_generation(model_path, prompts, max_length=150):
    """
    Evaluation function with improved scientific coherence and fact-checking.
    """
    generator = pipeline(
        'text-generation',
        model=model_path,
        tokenizer=model_path,
        device=0 if torch.cuda.is_available() else -1,
        do_sample=True,
        temperature=0.6,        # Reduced for more focused outputs
        top_p=0.85,            # Reduced for better coherence
        top_k=40,              # Reduced to limit vocabulary spread
        repetition_penalty=1.3, # Increased to prevent circular explanations
        num_return_sequences=5, # More candidates for better selection
        pad_token_id=50256
    )
    
    # Define valid scientific relationships and concepts
    scientific_concepts = {
        'mathematical': {
            'valid_concepts': ['differential equations', 'statistical patterns', 'regulatory networks', 'feedback loops'],
            'valid_relations': ['regulates', 'controls', 'influences', 'modulates']
        },
        'biological': {
            'valid_concepts': ['ATP synthesis', 'electron transport chain', 'oxidative phosphorylation', 'metabolic pathways'],
            'valid_relations': ['produces', 'converts', 'transforms', 'catalyzes']
        },
        'chemical': {
            'valid_concepts': ['reaction rate', 'transition state', 'binding site', 'molecular interactions'],
            'valid_relations': ['accelerates', 'stabilizes', 'reduces', 'facilitates']
        },
        'quantum': {
            'valid_concepts': ['coherence', 'superposition', 'tunneling', 'quantum states'],
            'valid_relations': ['affects', 'mediates', 'couples', 'interacts']
        }
    }
    
    def validate_scientific_statement(text, domain):
        """Validate scientific coherence of generated text"""
        score = 0
        if domain in scientific_concepts:
            concepts = scientific_concepts[domain]
            # Check for valid concept usage
            for concept in concepts['valid_concepts']:
                if concept in text.lower():
                    score += 2
            # Check for valid relationships
            for relation in concepts['valid_relations']:
                if relation in text.lower():
                    score += 1
            # Penalize mixing unrelated domains
            other_domains = set(scientific_concepts.keys()) - {domain}
            for other_domain in other_domains:
                if any(concept in text.lower() for concept in scientific_concepts[other_domain]['valid_concepts']):
                    score -= 2
        return score
    
    def construct_prompt(prompt):
        """Create a more focused scientific prompt"""
        domain = next((k for k in scientific_concepts.keys() if k in prompt.lower()), None)
        if domain:
            concepts = ' and '.join(scientific_concepts[domain]['valid_concepts'][:2])
            return f"Explain the scientific mechanism of how {prompt.lower()}, particularly considering {concepts}:"
        return f"Explain the scientific mechanism of how {prompt.lower()}:"
    
    def post_process(response):
        """Enhanced post-processing with scientific coherence checks"""
        # Remove text after incomplete sentences
        response = re.sub(r'[^.!?]+$', '', response)
        # Clean up parenthetical statements while preserving chemical formulas
        response = re.sub(r'\([^)]*\)', '', response)
        # Remove citation markers
        response = re.sub(r'\[\d+\]', '', response)
        # Clean up multiple spaces and newlines
        response = re.sub(r'\s+', ' ', response).strip()
        # Ensure sentences are complete
        sentences = response.split('.')
        complete_sentences = [s.strip() for s in sentences if len(s.strip().split()) > 3]
        return '. '.join(complete_sentences) + '.'
    
    print("Testing model on various scientific prompts...\n")
    
    for prompt in prompts:
        domain = next((k for k in scientific_concepts.keys() if k in prompt.lower()), None)
        structured_prompt = construct_prompt(prompt)
        
        # Generate multiple responses
        responses = generator(
            structured_prompt,
            max_length=max_length,
            num_return_sequences=5,
            pad_token_id=50256,
            truncation=True
        )
        
        # Process and score responses
        processed_responses = [(post_process(r['generated_text']), 
                              validate_scientific_statement(r['generated_text'], domain))
                             for r in responses]
        
        # Select best response based on scientific validity score
        best_response = max(processed_responses, key=lambda x: x[1])[0]
        
        print(f"Prompt: {prompt}\n{'-' * 50}")
        print(f"Generated Response:\n{best_response}\n")
        print("=" * 80 + "\n")
    
    return "Evaluation complete."

In [30]:
test_prompts = [
    "Mathematical patterns influence gene expression.",
    "Mitochondria produce energy in cells through biochemical processes.",
    "Catalysts reduce activation energy in chemical reactions.",
    "Quantum entanglement may influence biological processes.",
    "Quantum mechanics impacts molecular biology."
]


In [31]:
# Cell 4: Run Evaluation
evaluate_scientific_generation(MODEL_PATH, test_prompts)

Testing model on various scientific prompts...

Prompt: Mathematical patterns influence gene expression.
--------------------------------------------------
Generated Response:
Explain the scientific mechanism of how mathematical patterns influence gene expression. , particularly considering differential equations and statistical patterns: The fundamental idea behind this study is to identify a common underlying problem in biology. The main reason why such problems arise for many biologists, including some who have studied genetics or are interested in studying genetic variation among populations has been that they show no obvious causal relationship between biological processes. This hypothesis was first proposed by Albert Einstein's famous theory about relativity but it turned out to be incorrect as well because there were two important differences - one caused by quantum mechanics.


Prompt: Mitochondria produce energy in cells through biochemical processes.
-------------------------

'Evaluation complete.'